# AI 内容鉴真 - 权重下载与 ONNX 转换

在 Colab 中运行此笔记本，下载开源检测模型权重并转换为 ONNX 格式，然后下载回本地。

**输出文件**：
- `cnn_spotter.onnx` - CNNDetection (ResNet50)
- `universal_fake.onnx` - UniversalFakeDetect (CLIP ViT-L/14)
- `onnx_weights.zip` - 打包文件

## 1. 安装依赖

In [ ]:
!pip install -q torch torchvision onnx onnxruntime ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git
print("Dependencies installed.")

## 2. 下载 CNNDetection 权重 (ResNet50)

In [ ]:
import os
os.makedirs("weights", exist_ok=True)

# CNNDetection blur_jpg_prob0.5.pth (Dropbox 直链)
!wget -q "https://www.dropbox.com/s/2g2jagq2jn1fd0i/blur_jpg_prob0.5.pth?dl=1" -O weights/cnn_spotter.pth
!ls -lh weights/cnn_spotter.pth

## 3. 转换 CNNSpotter 为 ONNX

In [ ]:
import torch
import torchvision.models as models

# 构建 ResNet50，二分类输出
model = models.resnet50(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, 2)

# 加载权重
state_dict = torch.load("weights/cnn_spotter.pth", map_location="cpu")
if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
    state_dict = state_dict["model_state_dict"]
model.load_state_dict(state_dict)
model.eval()

# 导出 ONNX
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model, dummy, "weights/cnn_spotter.onnx",
    opset_version=17,
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    do_constant_folding=True,
)
print("CNNSpotter ONNX exported.")
!ls -lh weights/cnn_spotter.onnx

## 4. 验证 CNNSpotter ONNX

In [ ]:
import numpy as np
import onnxruntime as ort

sess = ort.InferenceSession("weights/cnn_spotter.onnx", providers=["CPUExecutionProvider"])
dummy = np.random.randn(1, 3, 224, 224).astype(np.float32)
result = sess.run(None, {"input": dummy})
print(f"Input shape: {dummy.shape}")
print(f"Output shape: {result[0].shape}")
print(f"Output logits: {result[0].flatten()}")
print("CNNSpotter ONNX verified OK.")

## 5. 下载 UniversalFakeDetect 分类头权重

In [ ]:
# 尝试从仓库 raw 下载分类头权重
import urllib.request
url = "https://raw.githubusercontent.com/Yuheng-Li/UniversalFakeDetect/main/pretrained_weights/fc_weights.pth"
try:
    urllib.request.urlretrieve(url, "weights/universal_fake.pth")
    print("UniversalFakeDetect weight downloaded.")
    !ls -lh weights/universal_fake.pth
except Exception as e:
    print(f"Download failed: {e}")
    print("Please manually download fc_weights.pth from the repo and upload to weights/")

## 6. 转换 UniversalFakeDetect 为 ONNX

In [ ]:
import os
if os.path.exists("weights/universal_fake.pth"):
    import torch
    import clip

    # 加载 CLIP ViT-L/14 视觉编码器
    clip_model, _ = clip.load("ViT-L/14", device="cpu")
    visual = clip_model.visual
    visual.eval()

    # 加载分类头
    classifier = torch.nn.Linear(visual.output_dim, 2)
    state = torch.load("weights/universal_fake.pth", map_location="cpu")
    if isinstance(state, dict):
        cls_state = {k.replace("fc.", "").replace("classifier.", ""): v
                     for k, v in state.items() if "fc" in k or "classifier" in k}
        if cls_state:
            classifier.load_state_dict(cls_state)
        else:
            classifier.load_state_dict(state)
    else:
        classifier.load_state_dict(state)
    classifier.eval()

    # 组合模型
    class CombinedModel(torch.nn.Module):
        def __init__(self, visual, classifier):
            super().__init__()
            self.visual = visual
            self.classifier = classifier
        def forward(self, x):
            return self.classifier(self.visual(x))

    model = CombinedModel(visual, classifier)
    model.eval()

    dummy = torch.randn(1, 3, 224, 224)
    torch.onnx.export(
        model, dummy, "weights/universal_fake.onnx",
        opset_version=17,
        input_names=["input"], output_names=["output"],
        dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
        do_constant_folding=True,
    )
    print("UniversalFake ONNX exported.")
    !ls -lh weights/universal_fake.onnx
else:
    print("Skipping: universal_fake.pth not found.")

## 7. 打包并下载

In [ ]:
import zipfile
import os

onnx_files = [f for f in os.listdir("weights") if f.endswith(".onnx")]
print(f"Found {len(onnx_files)} ONNX files: {onnx_files}")

with zipfile.ZipFile("onnx_weights.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in onnx_files:
        zf.write(f"weights/{f}", arcname=f)
        print(f"  + {f}")

!ls -lh onnx_weights.zip

# 自动下载到本地
from google.colab import files
files.download("onnx_weights.zip")
print("\nDownload started. Extract .onnx files to your local models/weights/ directory.")

## 8. 本地部署步骤

1. 解压 `onnx_weights.zip`
2. 将 `.onnx` 文件复制到项目的 `models/weights/` 目录
3. 修改 `.env`：
   ```
   ENABLED_MODELS=freq_detector,ela_detector,noise_detector,cfa_detector,color_detector,cnn_spotter,universal_fake
   ```
4. 重启服务：`docker compose restart gateway`